
# Real Data Pipeline - Data Quality Checks
## Comprehensive validation framework for GitHub data

In [0]:

# Purpose: Validate data quality at each layer



import pyspark.sql.functions as F
from datetime import datetime

catalog = "workspace"
schema = "github_analytics"

print("=" * 70)
print("DATA QUALITY VALIDATION FRAMEWORK")
print("=" * 70)


In [0]:

# Cell 1: Create Quality Metrics Table
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{schema}.quality_metrics (
    check_id STRING,
    check_name STRING,
    table_name STRING,
    layer STRING,
    check_type STRING,
    total_records INT,
    failed_records INT,
    pass_rate DOUBLE,
    severity STRING,
    check_timestamp TIMESTAMP,
    status STRING,
    error_message STRING
)
USING DELTA
""")

print("✅ Quality metrics table created")


In [0]:

# Cell 2: Bronze Layer Quality Checks
print("\n" + "=" * 70)
print("BRONZE LAYER QUALITY CHECKS")
print("=" * 70 + "\n")

bronze_repos = spark.table(f"{catalog}.{schema}.bronze_repositories")
bronze_contribs = spark.table(f"{catalog}.{schema}.bronze_contributors")

# Check 1: NULL values in critical fields
null_check_repos = bronze_repos.filter(
    F.col("repo_name").isNull() | 
    F.col("stars").isNull() | 
    F.col("language").isNull()
).count()

print(f"✅ Bronze Repos - NULL check: {null_check_repos} issues found")

# Check 2: Negative values (impossible for GitHub metrics)
negative_check = bronze_repos.filter(
    (F.col("stars") < 0) | 
    (F.col("forks") < 0) | 
    (F.col("watchers") < 0)
).count()

print(f"✅ Bronze Repos - Negative values check: {negative_check} issues")

# Check 3: Duplicate repositories
duplicate_check = bronze_repos.groupBy("repo_name").count().filter(F.col("count") > 1).count()

print(f"✅ Bronze Repos - Duplicates: {duplicate_check} duplicates")

# Check 4: Date validation (created_at before updated_at)
date_logic_check = bronze_repos.filter(
    F.col("created_at") > F.col("updated_at")
).count()

print(f"✅ Bronze Repos - Date logic: {date_logic_check} anomalies")

# Check 5: Contributors NULL check
contrib_null_check = bronze_contribs.filter(
    F.col("contributor_login").isNull() | 
    F.col("repo_name").isNull()
).count()

print(f"✅ Bronze Contributors - NULL check: {contrib_null_check} issues")

In [0]:

# Cell 3: Silver Layer Quality Checks
print("\n" + "=" * 70)
print("SILVER LAYER QUALITY CHECKS")
print("=" * 70 + "\n")

silver_repos = spark.table(f"{catalog}.{schema}.silver_repositories")
silver_contribs = spark.table(f"{catalog}.{schema}.silver_contributors")

# Check 1: Verify data quality flags
valid_repos = silver_repos.filter(F.col("is_valid") == True).count()
total_silver_repos = silver_repos.count()

print(f"✅ Silver Repos - Valid records: {valid_repos}/{total_silver_repos} ({round(valid_repos/total_silver_repos*100, 2)}%)")

# Check 2: Verify calculations
negative_popularity = silver_repos.filter(F.col("popularity_score") < 0).count()
print(f"✅ Silver Repos - Popularity score validation: {negative_popularity} invalid")

# Check 3: Days since calculations (should be >= 0)
negative_days = silver_repos.filter(
    (F.col("days_since_update") < 0) | 
    (F.col("days_since_created") < 0)
).count()

print(f"✅ Silver Repos - Days calculations: {negative_days} invalid")

# Check 4: Activity flag consistency
inactive_but_recent = silver_repos.filter(
    (F.col("is_active") == False) & 
    (F.col("days_since_update") <= 30)
).count()

print(f"✅ Silver Repos - Activity consistency: {inactive_but_recent} anomalies")


In [0]:

# Cell 4: Gold Layer Quality Checks
print("\n" + "=" * 70)
print("GOLD LAYER QUALITY CHECKS")
print("=" * 70 + "\n")
gold_repo_rankings = spark.table(f"{catalog}.{schema}.gold_repository_rankings")
gold_contrib_analysis = spark.table(f"{catalog}.{schema}.gold_contributor_analysis")
gold_ecosystem_health = spark.table(f"{catalog}.{schema}.gold_ecosystem_health")

# Check 1: Verify rankings are sequential
rankings = gold_repo_rankings.select("overall_rank").collect()
rank_list = [row[0] for row in rankings if row[0] is not None]
expected_ranks = set(range(1, len(rank_list) + 1))
actual_ranks = set(rank_list)

rank_gaps = len(expected_ranks - actual_ranks)
print(f"✅ Gold Rankings - Sequential check: {rank_gaps} gaps found")

# Check 2: Health score range (should be 0-100)
out_of_range = gold_ecosystem_health.filter(
    (F.col("health_score") < 0) | 
    (F.col("health_score") > 100)
).count()

print(f"✅ Gold Health Scores - Range validation: {out_of_range} out of range")

# Check 3: Contributor tier consistency
invalid_tiers = gold_contrib_analysis.filter(
    ~F.col("expertise_level").isin(["expert", "experienced", "beginner"])
).count()

print(f"✅ Gold Contributor Tiers - Enum validation: {invalid_tiers} invalid")

# Check 4: Repository tier consistency  
invalid_repo_tiers = gold_repo_rankings.filter(
    ~F.col("tier").isin(["platinum", "gold", "silver", "bronze"])
).count()

print(f"✅ Gold Repository Tiers - Enum validation: {invalid_repo_tiers} invalid")


In [0]:

# Cell 5: Cross-layer Consistency Checks
print("\n" + "=" * 70)
print("CROSS-LAYER CONSISTENCY CHECKS")
print("=" * 70 + "\n")

# Check 1: Row count progression (bronze >= silver >= gold)
bronze_count = spark.table(f"{catalog}.{schema}.bronze_repositories").count()
silver_count = spark.table(f"{catalog}.{schema}.silver_repositories").count()
gold_count = spark.table(f"{catalog}.{schema}.gold_repository_rankings").count()

print(f"✅ Row counts:")
print(f"   Bronze: {bronze_count}")
print(f"   Silver: {silver_count}")
print(f"   Gold: {gold_count}")

if bronze_count >= silver_count >= gold_count:
    print(f"   Status: ✅ Logical progression")
else:
    print(f"   Status: ⚠️ Unexpected progression")

# Check 2: Verify all bronze repos appear in silver
bronze_repos_df = spark.table(f"{catalog}.{schema}.bronze_repositories").select("repo_name")
silver_repos_df = spark.table(f"{catalog}.{schema}.silver_repositories").select("repo_name")

unmatched = bronze_repos_df.join(
    silver_repos_df,
    on="repo_name",
    how="anti"
).count()

print(f"\n✅ Bronze→Silver mapping: {unmatched} unmatched records")


In [0]:
# Cell 6: Freshness Checks
print("\n" + "=" * 70)
print("DATA FRESHNESS CHECKS")
print("=" * 70 + "\n")

# Check 1: Data is not stale
most_recent = bronze_repos.select(F.max("ingestion_timestamp")).collect()[0][0]
hours_old = (datetime.now() - most_recent).total_seconds() / 3600 if most_recent else None

print(f"✅ Most recent data: {most_recent}")
if hours_old and hours_old < 24:
    print(f"   Status: ✅ Fresh ({round(hours_old, 1)} hours old)")
else:
    print(f"   Status: ⚠️ Stale data")

# Check 2: All repositories have recent updates
stale_repos = silver_repos.filter(F.col("days_since_update") > 365).count()
print(f"\n✅ Stale repositories (> 1 year): {stale_repos}")
